# 1.01 Initializing Haunted_Places_Features_Added and Preliminary Data Exploration

We did the following to prepare our data:

## 1. Converting CSV to TSV,

**Create csv2tab script**:
1. To create Python Script Run in Terminal: touch csv2tab && chmod u+x csv2tab,
2. Paste 3 lines:,
    - #/usr/bin/env python,
    - import csv, sys,
    - csv.writer(sys.stdout, dialect='excel-tab').writerows(csv.reader(sys.stdin)),
2. Move csv2tab to script folder \./dsci_550_a1,
3. Activate Conda Environment and Run:,
    - ./dsci_550_a1/csv2tab < ./data/raw/haunted_places.csv > ./data/raw/haunted_places.tab,
4. delete

## 2. Preliminary Data Exploration and init haunted_places_features_added.tab
1. fill NAN values
    - missing descriptions filled as ''
    - City and locations filled by reading haunted description
    - Missing "longitude" and "latitudes" filled with "city_longitude" and "city_latitude" if they exist
    - Missing "city_longitude" and "city_latitudes" filled with "longitude" and "latitude" if they exist

    - 17 latitude and longitude values imputed using generic state latitude and longitudes

2. create "haunted_places_id" column

3. capitalize column names

### 3. Saving 

1. One copy is saved to *"./data/processed/haunted_places_features_added.tab"*. This tsv is **used for feature storage**.

2. One copy is saved to  *"./data/processed/haunted_places_features_cleaned.tab"*. This tsv is **used for feature extraction**




In [1]:
# System Path #
import os
import sys 

# Pandas, json and runtime #
import pandas as pd
import json

# Reading CSV
df = pd.read_csv("../data/raw/haunted_places.tab", sep = "\t")

df.head()


,city,country,description,location,state,state_abbrev,longitude,latitude,city_longitude,city_latitude
0,Ada,United States,Ada witch - Sometimes you can see a misty blue...,Ada Cemetery,Michigan,MI,-85.504893,42.962106,-85.495480,42.960727
1,Addison,United States,A little girl was killed suddenly while waitin...,North Adams Rd.,Michigan,MI,-84.381843,41.971425,-84.347168,41.986434
2,Adrian,United States,If you take Gorman Rd. west towards Sand Creek...,Ghost Trestle,Michigan,MI,-84.035656,41.904538,-84.037166,41.897547
3,Adrian,United States,"In the 1970's, one room, room 211, in the old ...",Siena Heights University,Michigan,MI,-84.017565,41.905712,-84.037166,41.897547
4,Albion,United States,Kappa Delta Sorority - The Kappa Delta Sororit...,Albion College,Michigan,MI,-84.745177,42.244006,-84.753030,42.243097


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10992 entries, 0 to 10991
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   city            10989 non-null  object 
 1   country         10992 non-null  object 
 2   description     10992 non-null  object 
 3   location        10989 non-null  object 
 4   state           10992 non-null  object 
 5   state_abbrev    10992 non-null  object 
 6   longitude       9731 non-null   float64
 7   latitude        9731 non-null   float64
 8   city_longitude  10963 non-null  float64
 9   city_latitude   10963 non-null  float64
dtypes: float64(4), object(6)
memory usage: 858.9+ KB


### Missing Values 
**We have 1272 rows with missing values**. 
- Values are in longitude and latitude columns
- Values are missing where local descriptors used 
    - *Hikyes Tomb, Hell's Bridge*
    - *Where the old Train Station used to be*


In [15]:
for col, count in df.isna().sum().items():
        print(f"[{col}]: {count}")

[city]: 3
[country]: 0
[description]: 0
[location]: 3
[state]: 0
[state_abbrev]: 0
[longitude]: 1261
[latitude]: 1261
[city_longitude]: 29
[city_latitude]: 29


In [2]:
## Fill Missing Description
df["description"] = df["description"].fillna('').astype(str)

## city and location ##

# Read Description
df.loc[4082, 'city'] = "Maumee"
# Looked up "michigan haunted theater 2005 and found it was in Grand Haven"
df.loc[176, 'city'] = "Grand Haven"
df.loc[176, 'city_longitude'] = 86.2284
df.loc[176, 'city_latitude'] = 43.0631

# Entry takes place in Florida. Description and were incorrectly separated.
df.loc[9465, 'city'] = "Miami"
df.loc[9465, 'description'] = df.loc[9465, 'description'] + df.loc[9465, 'location']
df.loc[9465, 'location'] = 'cemetary'

# Read Descriptions and filled location NAN Values
df.loc[2427, 'location'] = "Bodega Bay"
df.loc[2756, 'location'] = "Elsinore Middle School"
df.loc[4712, 'location'] = "Dolly Field"


## Longitude and Latitude ##

# Fill NAN Longitude values with city longitude
df.loc[df['longitude'].isna(), 'longitude'] = df[df['longitude'].isna()]['city_longitude']

# Fill NAN latitude values with city latitude
df.loc[df['latitude'].isna(), 'latitude'] = df[df['latitude'].isna()]['city_latitude']

# Fill NAN city_longitude values with longitude if it exists
df.loc[df['city_longitude'].isna(), 'city_longitude'] = df[df['city_longitude'].isna()]['longitude']

# Fill NAN city_latitude values with latitude if it exists
df.loc[df['city_latitude'].isna(), 'city_latitude'] = df[df['city_latitude'].isna()]['latitude']


## Filling last of coordinates with coordinates of state ##
state_coordinates_list = {
    'PA': {'Latitude': 41.2033, 'Longitude': -77.1945},
     'AR': { 'Latitude': 34.7465, 'Longitude': -92.2896},
     'AL': { 'Latitude': 32.3182, 'Longitude': -86.9023},
     'OH': { 'Latitude': 40.4173, 'Longitude': -82.9071},
     'ND': { 'Latitude': 47.5515, 'Longitude': -101.002},
     'ND': { 'Latitude': 47.5515, 'Longitude': -101.002},
     'KS': { 'Latitude': 39.0119, 'Longitude': -98.4842},
     'IL': { 'Latitude': 40.6331, 'Longitude': -89.3985},
     'IN': { 'Latitude': 40.2672, 'Longitude': -86.1349},
     'IN': { 'Latitude': 40.2672, 'Longitude': -86.1349},
     'VA': { 'Latitude': 37.4316, 'Longitude': -78.6569},
     'GA': { 'Latitude': 32.1656, 'Longitude': -82.9001},
     'FL': { 'Latitude': 27.9944, 'Longitude': -81.7603},
     'MN': { 'Latitude': 46.7296, 'Longitude': -94.6859}
}

for idx in df.loc[df['latitude'].isna()].index:
    state = df.loc[idx, 'state_abbrev']
    df.loc[idx, "latitude"] = state_coordinates_list[state]['Latitude']
    df.loc[idx, "longitude"] = state_coordinates_list[state]['Longitude']
    df.loc[idx, "city_longitude"] = state_coordinates_list[state]['Longitude']
    df.loc[idx, "city_latitude"] = state_coordinates_list[state]['Latitude']

for col, count in df.isna().sum().items():
        print(f"[{col}]: {count}")

[city]: 0
[country]: 0
[description]: 0
[location]: 0
[state]: 0
[state_abbrev]: 0
[longitude]: 0
[latitude]: 0
[city_longitude]: 0
[city_latitude]: 0


### Top Cities

Los angeles is the most haunted 

In [8]:
Top_Cities = df.groupby("city").count().sort_values("country", ascending = False).index.tolist()
print("Top 20 Cities:" , "\n" + "\n".join(Top_Cities[:20]))
df.groupby("city").count().sort_values("country", ascending = False).head(20)

Top 20 Cities: 
Los Angeles
San Antonio
Honolulu
Pittsburgh
Columbus
Salem
Springfield
El Paso
Houston
Laredo
Orlando
Riverside
Tucson
Chicago
Portland
Louisville
San Francisco
San Diego
Seattle
Baltimore


,country,description,location,state,state_abbrev,longitude,latitude,city_longitude,city_latitude
city,,,,,,,,,
Los Angeles,61,61,61,61,61,61,61,61,61
San Antonio,55,55,55,55,55,55,55,55,55
Honolulu,43,43,43,43,43,43,43,43,43
Pittsburgh,42,42,42,42,42,42,42,42,42
Columbus,41,41,41,41,41,41,41,41,41
Salem,40,40,40,40,40,40,40,40,40
Springfield,40,40,40,40,40,40,40,40,40
El Paso,38,38,38,38,38,38,38,38,38
Houston,34,34,34,34,34,34,34,34,34


### **Top States**:

California has most entries by a long-shot

In [9]:
Top_States = df.groupby("state").count().sort_values("country", ascending = False).index.tolist()
print("Top 20 States:" , "\n" + "\n".join(Top_States[:20]))
df.groupby("state").count().sort_values("country", ascending = False).head(20)

Top 20 States: 
California
Texas
Pennsylvania
Michigan
Ohio
New York
Illinois
Kentucky
Indiana
Massachusetts
Florida
Missouri
Georgia
Wisconsin
Alabama
Tennessee
Washington
Oklahoma
North Carolina
New Jersey


,city,country,description,location,state_abbrev,longitude,latitude,city_longitude,city_latitude
state,,,,,,,,,
California,1070,1070,1070,1070,1070,1070,1070,1070,1070
Texas,696,696,696,696,696,696,696,696,696
Pennsylvania,649,649,649,649,649,649,649,649,649
Michigan,529,529,529,529,529,529,529,529,529
Ohio,477,477,477,477,477,477,477,477,477
New York,459,459,459,459,459,459,459,459,459
Illinois,395,395,395,395,395,395,395,395,395
Kentucky,370,370,370,370,370,370,370,370,370
Indiana,351,351,351,351,351,351,351,351,351


## Capitalize Column Names and save to outfile


In [4]:
df["Haunted_Places_Id"] = df.index
cols = ['City',
 'Country',
 'Description',
 'Location',
 'State',
 'State_Abbrev',
 'Longitude',
 'Latitude',
 'City_Longitude',
 'City_Latitude',
 'Haunted_Places_Id']

df.columns = cols

outfile = "../data/processed/haunted_places_features_added.tab"

print(f"saving to {outfile}. This data is used for storing features")
df.to_csv(outfile, sep = "\t", index = False)
print("\n")
print(f"saved to ../data/processed/haunted_places_cleaned.tab. this tab is used for feature extraction")
df.to_csv("../data/processed/haunted_places_cleaned.tab", sep = "\t", index = False)




saving to ../data/processed/haunted_places_features_added.tab. This data is used for storing features


saved to ../data/processed/haunted_places_cleaned.tab. this tab is used for feature extraction
